# 🥉 Bronze Layer: The Raw Landing Zone

### 🎯 Objective
Verify raw data ingestion and understand the `bronze` branch role.

### 🌿 Branch Strategy: `bronze`
- **Purpose**: Immutable history of all raw data.
- **Why a separate branch?** 
  - Allows us to re-process data from scratch (Time Travel) if our transformation logic changes.
  - Isolates raw ingestion from production queries.
- **Nessie Reference**: `nessie.ecommerce.orders_bronze@bronze`

In [ ]:
import os
from pyspark.sql import SparkSession

# Correct Configuration for Production Environment
spark = SparkSession.builder \
    .appName("Bronze_Demo") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,software.amazon.awssdk:bundle:2.17.178,software.amazon.awssdk:url-connection-client:2.17.178") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "bronze") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://lakehouse/warehouse") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark Session Connected to Bronze Branch")

### 🔍 Inspect Raw Data
Reading strictly from the `bronze` branch.

In [ ]:
# Read from the bronze branch specifically
df_bronze = spark.sql("SELECT * FROM nessie.ecommerce.`orders_bronze@bronze` LIMIT 10")
df_bronze.show(truncate=False)

### 📊 Volume Check
Verifying the scale of our data (411M records).

In [ ]:
count = spark.sql("SELECT count(*) FROM nessie.ecommerce.`orders_bronze@bronze`").collect()[0][0]
print(f"Total Raw Records: {count:,}")